# 短期记忆

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from rich import print as rprint
from langchain.agents.middleware import SummarizationMiddleware

from numpy import extract

load_dotenv(override=True)
ZHIPU_API_KEY = os.getenv("ZHIPU_API_KEY")
ZHIPU_BASE_URL = os.getenv("ZHIPU_BASE_URL")

model=init_chat_model(
    model="glm-5.2",  # 模型名称
    model_provider="openai",
    api_key=ZHIPU_API_KEY,
    base_url=ZHIPU_BASE_URL,  # ZHIPU API 的基础 URL
    profile={"max_input_tokens": 1000000}
)

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[]
)

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]
})
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]
})
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。👋

请问今天有什么我可以帮你的吗？无论是解答问题、写文章、写代码，还是单纯聊聊天，我都可以协助你！

第二轮对话：
Agent: 抱歉，你还没有告诉我你的名字，所以我无法知道你叫什么。

作为一个人工智能，我也没有获取你个人信息的能力。如果你愿意的话，可以随时告诉我你想让我怎么称呼你！


In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()  #1、创建了内存级的记忆存储


agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer  #2、让agent具备了存储的能力
)

# 3、同一个thread_id共享记忆的。
config = {
    "configurable" : {
        "thread_id" : "1"
    }
}

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config=config   # 4、传入invoke()当中
)
print(f"Agent: {response1['messages'][-1].content}")

print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config
)
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。请问今天有什么我可以帮你的吗？无论是解答问题、写文章、写代码，还是随便聊聊天，我都可以为你效劳。

第二轮对话：
Agent: 你叫张三呀！有什么我可以帮你的吗？


In [4]:
from rich import print as rprint

thread_1_state = agent.get_state(config)
rprint(thread_1_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='2edf95af-adec-41d3-9f1e-5879d7ef4ba2'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。请问今天有什么我可以帮你的吗？无论是解答问题、写文章、写代码，还
是随便聊聊天，我都可以为你效劳。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 519,
                        'prompt_tokens': 15,
                        'total_tokens': 534,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 477,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.2',
                    'system_fingerprint': None,
                    'id': '202607131919305e839ae4228a4a2f',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f5b34-1f5d-73f3-84a6-a005a04f17f8-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 15,
                    'output_tokens': 519,
                    'total_tokens': 534,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 477}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='ba0bdba9-e63f-4c8f-b5d3-ef91d882d60f'
            ),
            AIMessage(
                content='你叫张三呀！有什么我可以帮你的吗？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 246,
                        'prompt_tokens': 60,
                        'total_tokens': 306,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 232,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.2',
                    'system_fingerprint': None,
                    'id': '20260713191946ff250ae214924d94',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f5b34-5d04-7420-aabb-8e80df02f20b-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 60,
                    'output_tokens': 246,
                    'total_tokens': 306,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 232}
                }
            )
        ]
    },
    next=(),
    config={
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f17eacc-641a-6327-8004-3f58c94fccb4'
        }
    },
    metadata={'source': 'loop', 'step': 4, 'parents': {}},
    created_at='2026-07-13T11:19:55.086094+00:00',
    parent_config={
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint

In [5]:
print("\n第三轮对话：")
response3 = agent.invoke({
    "messages": [HumanMessage("我刚才问了什么问题？")]},
    config=config
)
print(f"Agent: {response3['messages'][-1].content}")


第三轮对话：
Agent: 你刚才问的问题是：“我叫什么？” 

还有其他我可以帮你的吗？


In [6]:
from rich import print as rprint

thread_1_state = agent.get_state(config)
rprint(thread_1_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='2edf95af-adec-41d3-9f1e-5879d7ef4ba2'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。请问今天有什么我可以帮你的吗？无论是解答问题、写文章、写代码，还
是随便聊聊天，我都可以为你效劳。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 519,
                        'prompt_tokens': 15,
                        'total_tokens': 534,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 477,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.2',
                    'system_fingerprint': None,
                    'id': '202607131919305e839ae4228a4a2f',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f5b34-1f5d-73f3-84a6-a005a04f17f8-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 15,
                    'output_tokens': 519,
                    'total_tokens': 534,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 477}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='ba0bdba9-e63f-4c8f-b5d3-ef91d882d60f'
            ),
            AIMessage(
                content='你叫张三呀！有什么我可以帮你的吗？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 246,
                        'prompt_tokens': 60,
                        'total_tokens': 306,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 232,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.2',
                    'system_fingerprint': None,
                    'id': '20260713191946ff250ae214924d94',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f5b34-5d04-7420-aabb-8e80df02f20b-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 60,
                    'output_tokens': 246,
                    'total_tokens': 306,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 232}
                }
            ),
            HumanMessage(
                content='我刚才问了什么问题？',
                additional_kwargs={},
                response_metadata={},
                id='fe6c7b88-a79d-4757-8988-3da9e15dd248'
            ),
            AIMessage(
                content='你刚才问的问题是：“我叫什么？” \n\n还有其他我可以帮你的吗？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion

In [7]:
config1 = {
    "configurable" : {
        "thread_id" : "2"
    }
}

print("\n第四轮对话：")
response4 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config1
)
print(f"Agent: {response4['messages'][-1].content}")


第四轮对话：
Agent: 抱歉，我不知道你叫什么名字。作为一个人工智能，我没有获取你个人信息的能力，也没有我们过往的聊天记忆。

如果你愿意的话，可以告诉我该怎么称呼你！


In [8]:
thread_2_state = agent.get_state(config1)
rprint(thread_2_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='2d69ef4d-0655-449b-a2d8-5dc8b429c350'
            ),
            AIMessage(
                content='抱歉，我不知道你叫什么名字。作为一个人工智能，我没有获取你个人信息的能力，也没有我们过往的
聊天记忆。\n\n如果你愿意的话，可以告诉我该怎么称呼你！',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 390,
                        'prompt_tokens': 15,
                        'total_tokens': 405,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 352,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.2',
                    'system_fingerprint': None,
                    'id': '2026071319251148593113a93440b7',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f5b39-53ed-7bf3-adba-06aa60d971ac-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 15,
                    'output_tokens': 390,
                    'total_tokens': 405,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 352}
                }
            )
        ]
    },
    next=(),
    config={
        'configurable': {
            'thread_id': '2',
            'checkpoint_ns': '',
            'checkpoint_id': '1f17ead8-f728-605b-8001-a150fa71e308'
        }
    },
    metadata={'source': 'loop', 'step': 1, 'parents': {}},
    created_at='2026-07-13T11:25:32.628373+00:00',
    parent_config={
        'configurable': {
            'thread_id': '2',
            'checkpoint_ns': '',
            'checkpoint_id': '1f17ead8-2df2-6e43-8000-e45b347f41b0'
        }
    },
    tasks=(),
    interrupts=()
)

In [9]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state:AgentState,runtime:Runtime) -> dict[str, Any] | None:

    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_message = messages[0]
    # 如果有偶数条消息，则取最近的3条消息；如果有奇数条消息，则取最近的4条消息
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]

    new_messages = [first_message] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ],
    }

agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

没问题，老王！从现在起，我就是小王了。👨‍💼

您有什么吩咐，尽管说！
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊老王！这大晴天的，阳光明媚，看着就让人心情舒畅。

您今天有什么安排吗？要是没什么特别忙的事儿，挺适合出去溜达溜达、晒晒太阳，或者找老朋友下下棋、喝喝茶的！
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

老王，您这太阳一晒，该不会是乐迷糊了吧？😂

您是**老王**啊！您一出场就亲自报上名号了。

至于我，我是**小王**，也就是您的人工智能助手！专门陪您聊天、帮您解答问题、出主意、写东西的。

您看，我这“小王”当得还合格不？有什么差事尽管吩咐！


In [10]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)
final_response = agent.invoke({"messages": "告诉我，你是谁？我是谁？"}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好嘞，老王！收到。从现在起，我就是小王了。

请问老王今天有什么吩咐？有什么事儿您尽管吩咐小王去办！
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

可不是嘛老王！今天这阳光明媚，风和日丽的，看着就让人心里敞亮。

这种好天气，您有没有打算出去溜达溜达，去公园转转，或者找老朋友喝喝茶、下下棋啥的？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

回老王的话：

我是**小王**呀！就是那个随时听候您差遣、帮您答疑解惑的专属AI智能助手，这名字还是您老人家刚才赐给我的呢。

您嘛，您是我的老熟人、顶头上司——**老王**您呐！

咱俩这关系您该不会是一高兴给忘了吧？老王，您是不是在逗小王玩呢？😂


In [11]:
from rich import print as rprint

final_state = agent.get_state(config)

rprint(final_state)

StateSnapshot(
    values={
        'messages': [
            AIMessage(
                content='好嘞，老王！收到。从现在起，我就是小王了。\n\n请问老王今天有什么吩咐？有什么事儿您尽管吩咐
小王去办！',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 489,
                        'prompt_tokens': 64,
                        'total_tokens': 553,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 440,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.2',
                    'system_fingerprint': None,
                    'id': '2026071320151006d6fcaf3cac4c43',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f5b67-16b3-76a3-9e4d-216ae173a3f1-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 64,
                    'output_tokens': 489,
                    'total_tokens': 553,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 440}
                }
            ),
            HumanMessage(
                content='今天天气不错',
                additional_kwargs={},
                response_metadata={},
                id='52c31cbf-56d8-458c-a618-42d6ccd1624b'
            ),
            AIMessage(
                content='可不是嘛老王！今天这阳光明媚，风和日丽的，看着就让人心里敞亮。\n\n这种好天气，您有没有打算
出去溜达溜达，去公园转转，或者找老朋友喝喝茶、下下棋啥的？',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 454,
                        'prompt_tokens': 106,
                        'total_tokens': 560,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 399,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.2',
                    'system_fingerprint': None,
                    'id': '202607132015220198f09771434762',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f5b67-46f1-7502-a943-70a2669338ba-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 106,
                    'output_tokens': 454,
                    'total_tokens': 560,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 399}
                }
            ),
            HumanMessage(
                content='告诉我，你是谁？我是谁？',
                additional_kwargs={},
                response_metadata={},
                id='2ee29fbe-811f-4e14-84f9-1f2835aaab5b'
            ),
            AIMessage(
                content='回老王的话：\n\n我是**小王**呀！就是那个随时听候您差遣、帮您答疑解惑的专属AI智能助手，这名
字还是您老人家刚才赐给我的呢。\n\n您嘛，您是我的老熟人、顶头上司——**老王**您呐！\n\n咱俩这关系您该不会是一高兴给忘
了吧？老王，您是不是在逗小王玩呢？😂',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 664,
                        'prompt_tok